# 🚀 Guia Pré-Lançamento (Checklist de Produção)

Este documento contém as ações críticas e indispensáveis que devem ser realizadas **logo antes de lançar o sistema oficialmente** para os usuários finais. O foco é garantir a segurança, performance e blindagem dos dados da Centauro.

## 1. Desarmar a "Bomba" (Desativar o Endpoint de Seed)

Durante o desenvolvimento, usamos a rota `/api/seed_database` para limpar e popular o banco rapidamente. Em produção real, isso é um risco extremo.

- **Ação Imediata (Via Painel):** Acesse as variáveis de ambiente (`Variables`) no Railway e **apague** a variável `ALLOW_SEED` (ou altere seu valor para `false`). Isso travará o uso da rota bloqueando deleções acidentais.
- **Ação Definitiva (Via Código):** No arquivo `backend/app/main.py`, apague inteiramente o bloco de código correspondente ao endpoint `@app.post("/api/seed_database")`. Faça o commit e o *push* para remover o código permanentemente do servidor.

## 2. Restringir o CORS (Segurança de API)

Atualmente, nossa API aceita conexões de **qualquer lugar da internet** (isso ajudou na fase de testes com `localhost`). Ao lançar, precisamos restringir para que apenas o site oficial da Centauro possa pedir dados ao backend.

- **Ação:** No arquivo `backend/app/main.py`, encontre a configuração do CORS e mude de `allow_origins=["*"]` para aceitar apenas a sua URL real da Vercel. Exemplo:
```python
app.add_middleware(
    CORSMiddleware,
    allow_origins=["https://centauro-erp.vercel.app"], # <- ALTERE AQUI PARA SEU DOMINIO REAL
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)
```

## 3. Troca de Senhas e Limpeza de Contas de Teste

O script atual de banco de dados (`seed_data.py`) preencheu o sistema com dezenas de colaboradores fictícios e configurou as senhas de Administrador (incluindo o seu usuário `lucas.silva`) para o padrão `senha123`.

- **Ação 1:** Logue na plataforma como Administrador e altere imediatamente todas as senhas fracas para senhas fortes e exclusivas.
- **Ação 2:** Revise a tela de Colaboradores e exclua ou desative as pessoas e projetos fictícios que foram gerados se não forem ter utilidade real.

## 4. Variáveis de Ambiente, Logs e E-mails (Migração Outlook)

O arquivo `.env` nunca deve ser enviado ao GitHub, mas você deve assegurar que as variáveis em produção (nuvem) estejam blindadas:

- **MIGRAÇÃO GOOGLE -> OUTLOOK:** Atualmente estamos usando provedor Google (Gmail) para envio dos e-mails transacionais do sistema (`MAIL_USERNAME`). A empresa já migrou os e-mails corporativos para o **Outlook** e teremos a **desativação das contas do Google em aproximadamente 2 meses**. Lembre-se, como prioridade crítica, de atualizar o servidor SMTP no Railway (`MAIL_SERVER=smtp.office365.com` ou correspondente), a porta e a senha antes deste prazo para evitar a interrupção das notificações de sistema aos usuários.
- A chave API do Google (`GOOGLE_API_KEY`) no Railway deve pertencer a um projeto destinado apenas à produção (não misturar cotas de uso com desenvolvimento local).
- O e-mail emissor configurado é confiável e não exibe logs ou alertas confidenciais de dev/erro nas caixas visíveis pela diretoria ou clientes.

## 5. Backup do Banco de Dados

Antes que os funcionários reais da equipe da Centauro comecem a imputar dezenas de contratos, Notas Fiscais e Dados Sensíveis ao sistema:

- **Ação:** Acesse o painel do PostgreSQL do Railway e certifique-se de que as ferramentas de criação de Backups automáticos semanais/diários estão ativados (Isso previne dias de trabalho perdidos no caso de algum programador derrubar uma tabela acidentalmente).